In [253]:
from dotenv import load_dotenv
import os

load_dotenv() 

True

In [254]:
!pip install -q langchain-huggingface huggingface_hub langchain_core requests

In [255]:
from langchain_huggingface import HuggingFaceEndpoint,ChatHuggingFace
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

In [256]:
model = ChatHuggingFace(llm=HuggingFaceEndpoint(
    repo_id="deepseek-ai/DeepSeek-V4-Pro", 
    task="text-generation",
         
))

In [257]:
# tool create
from langchain_core.tools import InjectedToolArg
from typing import Annotated

@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
  """
  This function fetches the currency conversion factor between a given base currency and a target currency
  """
  url = f'https://v6.exchangerate-api.com/v6/3a7798064cc8aac60fd991c7/pair/{base_currency}/{target_currency}'

  response = requests.get(url)

  return response.json()

@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
  """
  given a currency conversion rate this function calculates the target currency value from a given base currency value
  """

  return base_currency_value * conversion_rate


In [258]:
convert.args

{'base_currency_value': {'title': 'Base Currency Value', 'type': 'integer'}}

In [259]:
get_conversion_factor.invoke({"base_currency": "USD", "target_currency": "INR"})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1779580802,
 'time_last_update_utc': 'Sun, 24 May 2026 00:00:02 +0000',
 'time_next_update_unix': 1779667202,
 'time_next_update_utc': 'Mon, 25 May 2026 00:00:02 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 95.8824}

In [260]:
convert.invoke({"base_currency_value": 10, "conversion_rate": 85.16})

851.5999999999999

In [261]:
# tool binding
llm_with_tools = model.bind_tools([get_conversion_factor, convert])

In [262]:
messages = [HumanMessage(content="What is the conversion factor between USD and INR, and based on that can you convert 10 USD to INR?")]

In [263]:
messages

[HumanMessage(content='What is the conversion factor between USD and INR, and based on that can you convert 10 USD to INR?', additional_kwargs={}, response_metadata={})]

In [264]:
ai_message = llm_with_tools.invoke(messages)

In [265]:
messages.append(ai_message)

In [266]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'USD', 'target_currency': 'INR'},
  'id': 'chatcmpl-tool-8257007fc24c7813',
  'type': 'tool_call'}]

In [ ]:
import json

for tool_call in ai_message.tool_calls:

    # Tool 1
    if tool_call["name"] == "get_conversion_factor":

        tool_message1 = get_conversion_factor.invoke(
            tool_call["args"]
        )

        conversion_rate = json.loads(
            tool_message1.content
        )["conversion_rate"]

        messages.append(tool_message1)

        # Manually call Tool 2
        tool_message2 = convert.invoke({
            "base_currency_value": 10,
            "conversion_rate": conversion_rate
        })

        messages.append(tool_message2)

messages

In [268]:
messages

[HumanMessage(content='What is the conversion factor between USD and INR, and based on that can you convert 10 USD to INR?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"base_currency": "USD", "target_currency": "INR"}', 'name': 'get_conversion_factor', 'description': None}, 'id': 'chatcmpl-tool-8257007fc24c7813', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 132, 'prompt_tokens': 393, 'total_tokens': 525}, 'model_name': 'deepseek-ai/DeepSeek-V4-Pro', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e5b25-dad8-7a71-a783-fd62a91f0a56-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'chatcmpl-tool-8257007fc24c7813', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 393, 'output_tokens': 132, 'total_tokens': 525}),
 ToolMessage(conten